In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType

# Read files without ID column
df_2020 = spark.read.csv("/Volumes/workspace/default/raw/attendance_2020.csv", header=True, inferSchema=True)
df_2021 = spark.read.csv("/Volumes/workspace/default/raw/attendance_2021.csv", header=True, inferSchema=True)
df_2023 = spark.read.csv("/Volumes/workspace/default/raw/attendance_2023.csv", header=True, inferSchema=True)

# Read files with ID column and drop it
df_2019 = spark.read.csv("/Volumes/workspace/default/raw/attendance_2019.csv", header=True, inferSchema=True).drop("ID")
df_2022 = spark.read.csv("/Volumes/workspace/default/raw/attendance_2022.csv", header=True, inferSchema=True).drop("ID")

# Combine all into one dataframe
df_attendance = df_2019.unionByName(df_2020).unionByName(df_2021).unionByName(df_2022).unionByName(df_2023)

df_enrolments = spark.read.csv(
    "/Volumes/workspace/default/raw/enrolments.csv",
    header=True,
    inferSchema=True
)

# ── ATTENDANCE ──────────────────────────────────────────
year_level_cols = [
    'Reception', 'Year_1', 'Year_2', 'Year_3', 'Year_4', 'Year_5', 'Year_6',
    'Primary_Other', 'Year_7', 'Year_8', 'Year_9', 'Year_10', 'Year_11',
    'Year_12', 'Secondary_Other'
]

df_att_silver = df_attendance \
    .drop('ID') \
    .withColumn('Census_Year', F.col('Census_Year').cast(IntegerType())) \
    .withColumn('School_Number', F.col('School_Number').cast(IntegerType())) \
    .withColumn('Latitude', F.col('Latitude').cast(DoubleType())) \
    .withColumn('Longitude', F.col('Longitude').cast(DoubleType()))


for col in year_level_cols:
    df_att_silver = df_att_silver.withColumn(col, F.col(col).cast(DoubleType()))

df_att_silver = df_att_silver.withColumn(
    'Avg_Attendance_Rate',
    (F.col('Reception') + F.col('Year_1') + F.col('Year_2') + F.col('Year_3') +
     F.col('Year_4') + F.col('Year_5') + F.col('Year_6') + F.col('Primary_Other') +
     F.col('Year_7') + F.col('Year_8') + F.col('Year_9') + F.col('Year_10') +
     F.col('Year_11') + F.col('Year_12') + F.col('Secondary_Other')) / len(year_level_cols)
)

df_att_silver.write \
    .format('delta') \
    .mode('overwrite') \
    .option("mergeSchema", "true") \
    .saveAsTable("silver_attendance")

df_attendance.show(5)   


# ── ENROLMENTS ──────────────────────────────────────────
enrolment_year_cols = [str(y) for y in range(2009, 2025)]

df_enr_silver = df_enrolments \
    .withColumnRenamed('Post_Code', 'Postcode') \
    .unpivot(
        ['School_Number', 'School_Name', 'School_Type', 'Suburb', 'Postcode', 'Latitude', 'Longitude'],
        enrolment_year_cols,
        'Year',
        'Total_Enrolments'
    ) \
    .withColumn('Year', F.col('Year').cast(IntegerType())) \
    .withColumn('Total_Enrolments', F.col('Total_Enrolments').cast(IntegerType()))


df_enr_silver.write \
    .format('delta') \
    .mode('overwrite') \
    .option("mergeSchema", "true") \
    .saveAsTable("silver_enrolments")

print("Successful")

# df_att_silver.printSchema()
# df_att_silver.show(5)

# df_enr_silver.printSchema()
# df_enr_silver.show(5)